In [7]:
import importlib, sys
# Force reload the module
importlib.reload(sys.modules['Class_definitions'])

<module 'Class_definitions' from 'C:\\Users\\dinab\\Desktop\\PhD Projects\\Ensemble methods\\GitHub_App\\medicaljourneys\\Class_definitions.py'>

In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from simulate_population import sim_population

In [9]:
from Class_definitions import (    prepare_data_simple_timeseries,    train_simple_timeseries,
    get_cindex_simple_timeseries,    SimpleBinaryTimeSeries)


In [22]:
#============================================================================
# HYPERPARAMETERS (matching Simple_Cox_with_NN-Copy1)
# ============================================================================
N_POPULTATON = 1000

BATCH_SIZE = 512
EPOCHS = 300
HIDDEN_LAYERS_COX = (64, 32)
HIDDEN_LAYERS_MULTIBINARY = (64, 32)
STEP_FORWARD = 2
N_STEPS = 5
N_INTERVALS = int(N_STEPS * STEP_FORWARD)  # time split every 2 years
COVARIATE_COLS = ["age_start", "bmi", "hyp", "smoke", "sex", "eth1", "eth2"]
EVENT_TYPES = ["a", "b", "c", "d", "e"]

# ============================================================================
# PART A: Generate Data
# ============================================================================
print("=" * 80)
print("PART A: Generating Simulation Data")
print("=" * 80)

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Generate population with 5 steps forward, step_forward=2
population = sim_population(N=N_POPULTATON, step_forward=STEP_FORWARD, randomseed=42)

for i in range(N_STEPS):
    population.step()
    print(f"Step {i+1}/{N_STEPS} completed")

# Get Cox format data (train/test split)
df_cox = population.to_cox_format()
print(f"Cox format data shape: {df_cox.shape}")
print(f"Columns: {df_cox.columns.tolist()}\n")

print("=" * 80)
print(f"Prepare the data for the time series; {len(df_cox)} x {N_INTERVALS} samples")
print("=" * 80)

df_train_long = prepare_data_simple_timeseries(    population,    features=COVARIATE_COLS,    event_types=EVENT_TYPES)
# 80/20 train/test split
n_train = int(0.8 * len(df_train_long))
df_train = df_train_long.iloc[:n_train].reset_index(drop=True)
df_test = df_train_long.iloc[n_train:].reset_index(drop=True)

print(f"\nTrain set: {len(df_train)} samples")
print(f"Test set: {len(df_test)} samples")


PART A: Generating Simulation Data
Step 1/5 completed
Step 2/5 completed
Step 3/5 completed
Step 4/5 completed
Step 5/5 completed
Cox format data shape: (1000, 18)
Columns: ['id', 'age_start', 'bmi', 'hyp', 'smoke', 'sex', 'eth1', 'eth2', 'event_a', 'time_a', 'event_b', 'time_b', 'event_c', 'time_c', 'event_d', 'time_d', 'event_e', 'time_e']

Prepare the data for the time series; 1000 x 10 samples

Train set: 4800 samples
Test set: 1200 samples


In [14]:
df_train_long

,id,interval,start_time,end_time,age_start,bmi,hyp,smoke,sex,eth1,eth2,event_a,event_b,event_c,event_d,event_e
0,0,0,0,1,62.1,-1.9,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
1,0,1,1,2,62.1,-1.9,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
2,0,2,2,3,62.1,-1.9,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
3,0,3,3,4,62.1,-1.9,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
4,0,4,4,5,62.1,-1.9,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9999,1,1,2,45.4,1.4,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
59996,9999,2,2,3,45.4,1.4,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
59997,9999,3,3,4,45.4,1.4,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0
59998,9999,4,4,5,45.4,1.4,0.0,0.0,1.0,0.0,0.0,0,0,0,0,0


In [6]:

# ===== LINEAR MODEL (NO HIDDEN LAYERS) =====
print("\n[1/3] Training Simple Binary TimeSeries (Linear)...")
model_linear = train_simple_timeseries(
    df_train,
    features=COVARIATE_COLS,
    event_types=EVENT_TYPES,
    hidden_dims=(),  # NO hidden layers
    lr=0.01,
    epochs=300,
    batch_size=512
)

cindex_linear = get_cindex_simple_timeseries(
    model_linear, df_test, COVARIATE_COLS, EVENT_TYPES
)

for e in EVENT_TYPES:
    cindex_df_list.append({
        'Model': 'Binary TimeSeries (Linear)',
        'Event': e,
        'C-Index': cindex_linear[e]
    })


# ===== WITH HIDDEN LAYERS =====
print("\n[2/3] Training Simple Binary TimeSeries (64-32)...")
model_hidden = train_simple_timeseries(
    df_train,
    features=COVARIATE_COLS,
    event_types=EVENT_TYPES,
    hidden_dims=(64, 32),  # Two hidden layers
    lr=0.01,
    epochs=300,
    batch_size=512
)

cindex_hidden = get_cindex_simple_timeseries(
    model_hidden, df_test, COVARIATE_COLS, EVENT_TYPES
)

for e in EVENT_TYPES:
    cindex_df_list.append({
        'Model': 'Binary TimeSeries (64-32)',
        'Event': e,
        'C-Index': cindex_hidden[e]
    })


# ===== WITH MORE LAYERS =====
print("\n[3/3] Training Simple Binary TimeSeries (128-64-32)...")
model_deep = train_simple_timeseries(
    df_train,
    features=COVARIATE_COLS,
    event_types=EVENT_TYPES,
    hidden_dims=(128, 64, 32),  # Three hidden layers
    lr=0.01,
    epochs=300,
    batch_size=512
)

cindex_deep = get_cindex_simple_timeseries(
    model_deep, df_test, COVARIATE_COLS, EVENT_TYPES
)

for e in EVENT_TYPES:
    cindex_df_list.append({
        'Model': 'Binary TimeSeries (128-64-32)',
        'Event': e,
        'C-Index': cindex_deep[e]
    })

# ===== RESULTS TABLE =====
cindex_df = pd.DataFrame(cindex_df_list)
cindex_pivot = cindex_df.pivot_table(index='Event', columns='Model', values='C-Index')
print("\n" + cindex_pivot.to_string())


[1/3] Training Simple Binary TimeSeries (Linear)...
Epoch 1/300, Loss: 0.8719
Epoch 51/300, Loss: 0.0000
Epoch 101/300, Loss: 0.0000
Epoch 151/300, Loss: 0.0000
Epoch 201/300, Loss: 0.0000
Epoch 251/300, Loss: 0.0000
Epoch 300/300, Loss: 0.0000


ZeroDivisionError: No admissable pairs in the dataset.